# Data vs Physics vs Hybrid

**Time: ~30 minutes**

Same ODE (`u' = -u`, `u(0) = 1`), solved three ways:
1. **Pure data** — train only on observed points
2. **Pure physics** — train only on the ODE (what we did in notebook 03)
3. **Hybrid** — combine data + physics

This is the "aha moment" — seeing exactly when and why physics helps.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

# Exact solution for comparison
def exact_solution(t):
    return np.exp(-t)

## Setup: Sparse, Noisy Data

We simulate a realistic scenario: only 10 noisy measurements in `[0, 1]`, but we want the solution over `[0, 3]`.

In [ ]:
# "Observed" data: sparse and noisy, only in [0, 1]
np.random.seed(42)
t_data_np = np.sort(np.random.uniform(0, 1, 10))
u_data_np = np.exp(-t_data_np) + 0.03 * np.random.randn(10)  # 3% noise

t_data = torch.tensor(t_data_np, dtype=torch.float32).unsqueeze(1)
u_data = torch.tensor(u_data_np, dtype=torch.float32).unsqueeze(1)

# Evaluation domain: [0, 3] — extrapolation beyond the data!
t_eval = torch.linspace(0, 3, 300).unsqueeze(1)

plt.figure(figsize=(8, 4))
plt.plot(t_eval.numpy(), exact_solution(t_eval.numpy()), 'b-', label='Exact', linewidth=2)
plt.scatter(t_data_np, u_data_np, c='red', s=60, zorder=5, label='Observed (noisy)')
plt.axvline(x=1, color='gray', linestyle='--', alpha=0.5, label='Data boundary')
plt.xlabel('t'); plt.ylabel('u(t)')
plt.title('The challenge: 10 noisy points in [0,1] → predict [0,3]')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Helper: create a fresh model and train it
def make_model():
    return nn.Sequential(
        nn.Linear(1, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 1),
    )

def train_model(model, loss_fn, n_epochs=5000, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        loss = loss_fn(model)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return history

## Approach 1: Pure Data

Just fit the 10 observed points. No physics at all.

In [ ]:
model_data = make_model()

def data_only_loss(model):
    return torch.mean((model(t_data) - u_data)**2)

hist_data = train_model(model_data, data_only_loss)
print(f"Final data loss: {hist_data[-1]:.4e}")

## Approach 2: Pure Physics

Only the ODE residual + initial condition. No observed data.

In [ ]:
model_physics = make_model()

t_colloc = torch.linspace(0, 3, 100).unsqueeze(1).requires_grad_(True)
t_ic = torch.zeros(1, 1)
u_ic = torch.ones(1, 1)

def physics_only_loss(model):
    # ODE residual
    u = model(t_colloc)
    du_dt = torch.autograd.grad(u, t_colloc, torch.ones_like(u), create_graph=True)[0]
    residual = du_dt + u
    loss_physics = torch.mean(residual**2)
    # IC
    loss_ic = torch.mean((model(t_ic) - u_ic)**2)
    return loss_physics + 10.0 * loss_ic

hist_physics = train_model(model_physics, physics_only_loss)
print(f"Final physics loss: {hist_physics[-1]:.4e}")

## Approach 3: Hybrid (Data + Physics)

Combine data fitting and physics — the best of both worlds.

In [ ]:
model_hybrid = make_model()

def hybrid_loss(model):
    # Data loss
    loss_data = torch.mean((model(t_data) - u_data)**2)
    # ODE residual
    u = model(t_colloc)
    du_dt = torch.autograd.grad(u, t_colloc, torch.ones_like(u), create_graph=True)[0]
    residual = du_dt + u
    loss_physics = torch.mean(residual**2)
    # IC
    loss_ic = torch.mean((model(t_ic) - u_ic)**2)
    return loss_data + loss_physics + 10.0 * loss_ic

hist_hybrid = train_model(model_hybrid, hybrid_loss)
print(f"Final hybrid loss: {hist_hybrid[-1]:.4e}")

## The Comparison

In [ ]:
with torch.no_grad():
    pred_data = model_data(t_eval).numpy()
    pred_physics = model_physics(t_eval).numpy()
    pred_hybrid = model_hybrid(t_eval).numpy()

u_true = exact_solution(t_eval.numpy())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, pred, title, color in [
    (axes[0], pred_data, 'Pure Data', 'green'),
    (axes[1], pred_physics, 'Pure Physics', 'orange'),
    (axes[2], pred_hybrid, 'Hybrid', 'purple'),
]:
    ax.plot(t_eval.numpy(), u_true, 'b-', label='Exact', linewidth=2)
    ax.plot(t_eval.numpy(), pred, '--', color=color, label='Predicted', linewidth=2)
    ax.scatter(t_data_np, u_data_np, c='red', s=40, zorder=5, label='Data')
    ax.axvline(x=1, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('t'); ax.set_ylabel('u(t)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.3, 1.3)

plt.suptitle('Same ODE, Three Approaches', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Quantitative comparison
for name, pred in [("Data only", pred_data), ("Physics only", pred_physics), ("Hybrid", pred_hybrid)]:
    rel_l2 = np.linalg.norm(pred - u_true) / np.linalg.norm(u_true)
    # Error in extrapolation region (t > 1)
    mask = t_eval.numpy().flatten() > 1.0
    rel_l2_extrap = np.linalg.norm(pred[mask] - u_true[mask]) / np.linalg.norm(u_true[mask])
    print(f"{name:15s} | Full rel-L2: {rel_l2:.4f} | Extrapolation rel-L2: {rel_l2_extrap:.4f}")

## What We Learned

| Approach | In data range | Extrapolation | Key insight |
|----------|--------------|---------------|-------------|
| **Pure data** | Good fit | Diverges wildly | Networks extrapolate poorly |
| **Pure physics** | Good everywhere | Good | The ODE constrains the solution globally |
| **Hybrid** | Best fit | Good | Data improves local accuracy, physics ensures global consistency |

**The takeaway:** Physics acts as a regularizer that constrains the solution to be physically plausible, even in regions with no data. This is why PINNs are especially powerful for:
- Sparse data scenarios
- Extrapolation beyond observed regions
- Noisy data (physics smooths out noise)

## Loss History Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(hist_data, label='Data only', alpha=0.8)
ax.semilogy(hist_physics, label='Physics only', alpha=0.8)
ax.semilogy(hist_hybrid, label='Hybrid', alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Loss Comparison')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Exercises

1. **More data**: Increase to 50 data points. Does pure data catch up?
2. **More noise**: Increase noise to 10%. How does the hybrid model handle it?
3. **Data in full domain**: Give data over `[0, 3]` instead of `[0, 1]`. Now does pure data work?
4. **Harder ODE**: Try `u' = -5u` (fast decay). Which approach handles it best?

## What's Next

**Notebook 05** steps up from ODEs to PDEs: we'll solve Burgers' equation with spatial derivatives and boundary conditions.